In [1]:
import torch
from torch import nn

> 不同之处

卷积核的权重是可学习的参数, 并不是固定写死的.

当卷积层执行严格卷积运算(水平和垂直翻转二维卷积核张量后, 再互相关运算)时，将得到与互相关运算中相同的输出。

每个通道不是独立学习的，而是为了共同使用而优化的。因此，多输出通道并不仅是学习多个单通道的检测器。`

# 计算输出大小

卷积和池化都一样计算

> 尺寸

当输入高度和宽度两侧的填充数量分别为$p_h$和$p_w$时，我们称之为填充$(p_h, p_w)$。当$p_h = p_w = p$时，填充是$p$。

同理，当高度和宽度上的步幅分别为$s_h$和$s_w$时，我们称之为步幅$(s_h, s_w)$。特别地，当$s_h = s_w = s$时，我们称步幅为$s$。


在实践中，我们很少使用不一致的步幅或填充，也就是说，我们通常有$p_h = p_w$和$s_h = s_w$。

> 默认值

- 卷积: **填充为0，步幅为1。**

- 池化: **填充为0，步幅与kernel_size的大小相同。**

> 公式

假设输入形状为$n_h\times n_w$，卷积核形状为$k_h\times k_w$，填充$p_h \times p_w$, 垂直步幅为$s_h \times s_w$(上下一半,左右一半)时, 输出形状为

$$\lfloor(n_h-k_h+p_h+s_h)/s_h\rfloor \times \lfloor(n_w-k_w+p_w+s_w)/s_w\rfloor.$$


特殊例子:
- 没有填充p=1, 步幅s=1时,
    
    $$\lfloor(n_h-k_h+0+1)/1\rfloor \times \lfloor(n_w-k_w+0+1)/1\rfloor.$$
    即$$(n_h-k_h+1) \times (n_w-k_w+1)$$
    
- 有填充, 步幅s=1时
  
    $$(n_h-k_h+p_h+1)\times(n_w-k_w+p_w+1)$$

- 特殊填充$p_h=k_h-1$和$p_w=k_w-1$.(使输入和卷积输出具有相同的高度和宽度), 步幅为1时

    $$(n_h)\times(n_w)$$
    

- 特殊填充$p_h=k_h-1$和$p_w=k_w-1$.(使输入和卷积输出具有相同的高度和宽度)时

    $$\lfloor(n_h+s_h-1)/s_h\rfloor \times \lfloor(n_w+s_w-1)/s_w\rfloor$$

- 特殊填充$p_h=k_h-1$和$p_w=k_w-1$.(使输入和卷积输出具有相同的高度和宽度), 且输入的高度和宽度可以被垂直和水平步幅整除时

    $$\lfloor(n_h+s_h-1)/s_h\rfloor \times \lfloor(n_w+s_w-1)/s_w\rfloor.$$
    即$$(n_h/s_h+\lfloor(s_h-1)/s_h\rfloor) \times (n_w/s_w+\lfloor(s_h-1)/s_h)$$
    又$\lfloor(s_h-1)/s_h\rfloor=0$, 则
    $$(n_h/s_h) \times (n_w/s_w)$$

# 卷积

## 默认卷积

对于任何单个卷积，我们可能只会丢失几个像素, 但随着我们应用许多连续卷积层，累积丢失的像素数就多了。比如，一个$240 \times 240$像素的图像，经过$10$层$5 \times 5$的卷积后，将减少到$200 \times 200$像素。

In [5]:
# 这里的 `1，1` 表示批量大小和通道数都是1
net = nn.Conv2d(1, 1, kernel_size=3)
X = torch.rand(1, 1, 4, 4)
net(X).shape

torch.Size([1, 1, 2, 2])

## 填充

- 假设$k_h$是奇数，我们将在高度的两侧填充$p_h/2$行。

- 假设$k_h$是偶数，则一种可能性是在输入顶部填充$\lceil p_h/2\rceil$行，在底部填充$\lfloor p_h/2\rfloor$行。



使用奇数的核大小提供了便利, 我们可以在顶部和底部填充相同数量的行，在左侧和右侧填充相同数量的列。例如1,3,5,7.

In [7]:
# 这里的padding是上下左右各填充1, 所以p=2, 则(n-k+2*p+1)
# 如果要保持不变, 那就是 padding = (kernel_size -1)/2
# 这里的 `1，1` 表示批量大小和通道数都是1
net = nn.Conv2d(1, 1, kernel_size=3, padding=1)
X = torch.rand(1, 1, 4, 4)
net(X).shape

torch.Size([1, 1, 4, 4])

当卷积核的高度和宽度不同时，我们可以填充不同的高度和宽度.

In [66]:
# 这里的padding是上下各填充2, 左右各填充1
net = nn.Conv2d(1, 1, kernel_size=(5, 3), padding=(2, 1))
X = torch.rand(1, 1, 4, 4)
net(X).shape

torch.Size([1, 1, 4, 4])

## 步幅

In [11]:
net = nn.Conv2d(1, 1, kernel_size=3, padding=1, stride=2)
X = torch.rand(1, 1, 4, 4)
net(X).shape

torch.Size([1, 1, 2, 2])

In [14]:
net = nn.Conv2d(1, 1, kernel_size=(3, 5), padding=(0, 1), stride=(3, 4))
X = torch.rand(1, 1, 4, 4)
net(X).shape

torch.Size([1, 1, 1, 1])

## 通道

### 多输入通道,单输出通道

![两个输入通道的互相关计算。](../img/conv-multi-in.svg)

In [30]:
# 输入1层2通道卷积核, 输出1层卷积结果(相加)
net = nn.Conv2d(2, 1, kernel_size=2)
# 二通道的图像
X = torch.randn(2, 3, 3)
net(X).shape

torch.Size([1, 2, 2])

### 多输入通道,多输出通道

In [31]:
# 输入3层2通道卷积核, 输出3层卷积结果
net = nn.Conv2d(2, 3, kernel_size=2)
X = torch.randn(2, 3, 3)
net(X).shape

torch.Size([3, 2, 2])

![](../img/conv-1x1.svg)
互相关计算使用了具有3个输入通道和2个输出通道的 $1\times 1$ 卷积核。意思是输入2层3通道卷积核, 输出2层卷积结果. 图中, 核函数分为上下两层, 浅蓝色的上层3通道核函数和输入图像的浅蓝色三通道像素卷积, 得到的是输出的第一通道浅蓝色卷积结果；深蓝色的下层3通道核函数和输入图像的深蓝色三通道像素卷积, 得到的是输出的第二通道深蓝色卷积结果. 

这里输入和输出具有相同的高度和宽度，输出中的每个元素都是从输入图像中同一位置的元素的线性组合。我们可以将$1\times 1$卷积层看作是在每个像素位置应用的全连接层，以$c_i$个输入值转换为$c_o$个输出值。同时，$1\times 1$卷积层需要的权重维度为$c_o\times c_i$，再额外加上一个偏置。

In [27]:
# 输入2层3通道卷积核, 输出2层卷积结果
net = nn.Conv2d(3, 2, kernel_size=1)
X = torch.randn(3, 3, 3)
net(X).shape

torch.Size([2, 3, 3])

# 池化

我们通常计算汇聚窗口中所有元素的最大值或平均值。这些操作分别称为最大汇聚层（maximum pooling）和平均汇聚层（average pooling)

## 最大汇聚层

In [22]:
# net = nn.MaxPool2d(kernel_size=2, stride=2) 
net = nn.MaxPool2d(2)
#  non-empty 3D or 4D (batch mode) tensor expected for input
# reshape((1, 1, 4, 4)), 样本数1, 通道数1
# reshape(1, 3, 3), 通道数1
X = torch.arange(9, dtype=torch.float32).reshape(1, 3, 3)
X, net(X)

(tensor([[[0., 1., 2.],
          [3., 4., 5.],
          [6., 7., 8.]]]),
 tensor([[[4.]]]))

In [73]:
net = nn.MaxPool2d(2, stride=1) 
X = torch.arange(9, dtype=torch.float32).reshape(1, 3, 3)
net(X)

tensor([[[4., 5.],
         [7., 8.]]])

![](../img/pooling.svg)

In [74]:
# padding 也是上下各填充
net = nn.MaxPool2d((2, 3), padding=(1, 0), stride=1)
X = torch.arange(8, dtype=torch.float32).reshape(1, -1, 4)
X, net(X)

(tensor([[[0., 1., 2., 3.],
          [4., 5., 6., 7.]]]),
 tensor([[[2., 3.],
          [6., 7.],
          [6., 7.]]]))

在处理多通道输入数据时，汇聚层在每个输入通道上单独运算，而不是像卷积层一样在通道上对输入进行汇总。 这意味着汇聚层的输出通道数与输入通道数相同。

In [76]:
net = nn.MaxPool2d(2, stride=1)
# 两个通道
X = torch.arange(12, dtype=torch.float32).reshape(2, -1, 3)
X, net(X)

(tensor([[[ 0.,  1.,  2.],
          [ 3.,  4.,  5.]],
 
         [[ 6.,  7.,  8.],
          [ 9., 10., 11.]]]),
 tensor([[[ 4.,  5.]],
 
         [[10., 11.]]]))

## 平均汇聚层

In [80]:
net = nn.AvgPool2d(2, stride=1)
X = torch.arange(9, dtype=torch.float32).reshape(1, 3, 3)
X, net(X)

(tensor([[[0., 1., 2.],
          [3., 4., 5.],
          [6., 7., 8.]]]),
 tensor([[[2., 3.],
          [5., 6.]]]))